# KoACD Linguistic Feature Analysis

이 노트북은 KoACD 공식 GitHub 저장소의 6개 CSV 파일을 직접 불러와 분석합니다.

- 파일 업로드 필요 없음
- Colab 기본 sample_data 읽지 않음
- UTF-8, CP949, EUC-KR 인코딩 자동 처리
- 실행 후 `KoACD_analysis_results.zip` 자동 생성


In [ ]:
# ============================================================
# KoACD Linguistic Feature Analysis - Direct GitHub Version v2
# 생성 방식 및 LLM 모델별 언어 특성 차이 분석


In [ ]:
# ============================================================

# 이 버전은 파일 업로드가 필요 없습니다.
# KoACD 공식 GitHub 저장소의 6개 CSV를 직접 불러와 분석합니다.
# v2 수정점: GitHub CSV 인코딩이 UTF-8이 아닐 때 cp949/euc-kr도 자동 시도합니다.

import os
import re
import io
import zipfile
import warnings
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, kruskal
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
RESULT_DIR = Path("/content/KoACD_analysis_results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("설정 완료")
print("결과 폴더:", RESULT_DIR)




In [ ]:
# ============================================================
# 1. KoACD 공식 GitHub CSV 직접 불러오기


In [ ]:
# ============================================================

BASE_RAW_URL = "https://raw.githubusercontent.com/cocoboldongle/KoACD/main"

file_info = [
    {"file": "Cognitive_Balancing_Claude_Data.csv", "method": "Balancing", "model": "Claude"},
    {"file": "Cognitive_Balancing_Gemini_Data.csv", "method": "Balancing", "model": "Gemini"},
    {"file": "Cognitive_Balancing_Gpt_Data.csv", "method": "Balancing", "model": "GPT"},
    {"file": "Cognitive_Clarification_Claude_Data.csv", "method": "Clarification", "model": "Claude"},
    {"file": "Cognitive_Clarification_Gemini_Data.csv", "method": "Clarification", "model": "Gemini"},
    {"file": "Cognitive_Clarification_Gpt_Data.csv", "method": "Clarification", "model": "GPT"},
]


def read_csv_from_url(url):
    """
    GitHub raw CSV를 bytes로 받은 뒤 여러 인코딩을 시도해 읽는다.
    KoACD CSV가 UTF-8이 아닐 수 있으므로 cp949/euc-kr도 시도한다.
    """
    with urlopen(url) as response:
        raw_bytes = response.read()

    encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr", "latin1"]
    last_error = None

    for enc in encodings:
        try:
            text = raw_bytes.decode(enc)
            df = pd.read_csv(io.StringIO(text))
            print(f"  - 인코딩 성공: {enc}, shape={df.shape}")
            return df
        except Exception as e:
            last_error = e

    raise last_error


dfs = []

for item in file_info:
    url = f"{BASE_RAW_URL}/{item['file']}"
    print("읽는 중:", item["file"])

    df = read_csv_from_url(url)

    df["source_file"] = item["file"]
    df["Generation Method"] = item["method"]
    df["Source File Model"] = item["model"]

    dfs.append(df)

raw_df = pd.concat(dfs, ignore_index=True)

print("\n불러오기 완료")
print("전체 데이터 크기:", raw_df.shape)
print("컬럼 목록:")
for i, col in enumerate(raw_df.columns):
    print(i, ":", repr(col))

display(raw_df.head(3))

if len(raw_df) != 108717:
    print("\n주의: 전체 행 수가 108,717이 아닙니다.")
    print("현재 행 수:", len(raw_df))
    print("GitHub 원본 데이터가 갱신되었거나 일부 파일을 다르게 읽었을 수 있습니다.")
else:
    print("\n행 수 확인 완료: 108,717개")




In [ ]:
# ============================================================
# 2. 주요 컬럼 자동 탐색


In [ ]:
# ============================================================

def normalize_col_name(name):
    return str(name).lower().replace(" ", "").replace("_", "").replace("-", "")


def find_column_soft(df, candidates):
    for cand in candidates:
        if cand in df.columns:
            return cand

    normalized_map = {normalize_col_name(c): c for c in df.columns}
    for cand in candidates:
        key = normalize_col_name(cand)
        if key in normalized_map:
            return normalized_map[key]

    return None


TEXT_COL = find_column_soft(
    raw_df,
    [
        "Generated Story", "GeneratedStory", "generated_story", "Generated_Story",
        "story", "Story", "text", "Text", "sentence", "Sentence",
        "content", "Content", "question", "Question", "response", "Response",
        "original_text", "Original Text", "clarified_text", "balanced_text",
        "input", "Input", "output", "Output",
    ],
)

LABEL_COL = find_column_soft(
    raw_df,
    [
        "Cognitive Distortion", "CognitiveDistortion",
        "Cognitive Distortion Type", "CognitiveDistortionType",
        "cognitive_distortion", "cognitive_distortion_type",
        "distortion", "Distortion", "distortion_type", "Distortion Type",
        "label", "Label", "category", "Category", "type", "Type",
    ],
)

METHOD_COL = "Generation Method"
MODEL_COL = "Source File Model"


if TEXT_COL is None:
    object_cols = raw_df.select_dtypes(include="object").columns.tolist()
    object_cols = [
        c for c in object_cols
        if c not in ["source_file", "Generation Method", "Source File Model"]
    ]

    if not object_cols:
        raise ValueError("텍스트로 보이는 문자열 컬럼이 없습니다.")

    avg_lengths = {col: raw_df[col].astype(str).str.len().mean() for col in object_cols}

    print("\n텍스트 컬럼 후보 평균 길이:")
    for col, length in sorted(avg_lengths.items(), key=lambda x: x[1], reverse=True):
        print(col, ":", round(length, 2))

    TEXT_COL = max(avg_lengths, key=avg_lengths.get)
    print("\nTEXT_COL 자동 추정:", TEXT_COL)


if LABEL_COL is None:
    distortion_keywords = [
        "All-or-Nothing", "Overgeneralization", "Mental Filtering",
        "Discounting", "Jumping", "Magnification", "Emotional Reasoning",
        "Should", "Labeling", "Personalization",
        "흑백사고", "과잉 일반화", "부정적 필터링", "긍정 축소화",
        "성급한 판단", "확대와 축소", "감정적 추론",
        "해야 한다 진술", "낙인찍기", "개인화"
    ]

    best_col = None
    best_score = -1

    for col in raw_df.select_dtypes(include="object").columns:
        if col in ["source_file", "Generation Method", "Source File Model", TEXT_COL]:
            continue

        values = raw_df[col].dropna().astype(str)
        if values.empty:
            continue

        unique_count = values.nunique()
        if unique_count > 50:
            continue

        sample_values = " ".join(values.drop_duplicates().head(50).tolist())
        score = sum(keyword in sample_values for keyword in distortion_keywords)

        if score > best_score:
            best_score = score
            best_col = col

    if best_col is not None and best_score > 0:
        LABEL_COL = best_col
        print("\nLABEL_COL 자동 추정:", LABEL_COL)
    else:
        print("\n라벨 컬럼 자동 탐색 실패")
        print("컬럼 목록:", raw_df.columns.tolist())
        raise ValueError("인지왜곡 라벨 컬럼을 찾지 못했습니다.")


print("\n최종 사용 컬럼")
print("TEXT_COL:", TEXT_COL)
print("LABEL_COL:", LABEL_COL)
print("METHOD_COL:", METHOD_COL)
print("MODEL_COL:", MODEL_COL)

print("\n라벨 값 예시:")
print(raw_df[LABEL_COL].dropna().astype(str).value_counts().head(20))




In [ ]:
# ============================================================
# 3. 텍스트 전처리 및 분석 데이터 정리


In [ ]:
# ============================================================

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"[-=]{3,}", " ", text)
    text = re.sub(r"(성별|나이|학년)\s*[:：]\s*\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


analysis_df = raw_df.copy()

analysis_df["clean_text"] = analysis_df[TEXT_COL].apply(clean_text)
analysis_df = analysis_df[analysis_df["clean_text"].str.len() > 0].copy()

analysis_df[METHOD_COL] = analysis_df[METHOD_COL].astype(str).str.strip()
analysis_df[MODEL_COL] = analysis_df[MODEL_COL].astype(str).str.strip()
analysis_df[LABEL_COL] = analysis_df[LABEL_COL].astype(str).str.strip()

analysis_df = analysis_df[analysis_df[METHOD_COL].isin(["Balancing", "Clarification"])].copy()
analysis_df = analysis_df[analysis_df[MODEL_COL].isin(["Claude", "GPT", "Gemini"])].copy()

if analysis_df.empty:
    raise ValueError("최종 분석 데이터가 비었습니다. 데이터 구조를 확인해야 합니다.")

print("\n전처리 후 최종 분석 데이터 크기:", analysis_df.shape)

print("\n생성 방식 분포")
print(analysis_df[METHOD_COL].value_counts())

print("\nLLM 모델 분포")
print(analysis_df[MODEL_COL].value_counts())




In [ ]:
# ============================================================
# 4. 단어 목록 및 언어 특성 변수 생성


In [ ]:
# ============================================================

negative_words = [
    "안", "못", "없", "싫", "불안", "걱정", "두렵", "무섭",
    "힘들", "괴롭", "우울", "슬프", "짜증", "화", "외롭",
    "불행", "포기", "실패", "문제", "나쁘",
]

emotion_words = [
    "기쁘", "행복", "좋", "즐겁", "슬프", "우울", "불안",
    "화나", "분노", "외롭", "답답", "무섭", "두렵", "걱정",
    "짜증", "괴롭", "서럽", "억울", "상처", "힘들",
]

extreme_words = [
    "항상", "절대", "전부", "모두", "아무도", "완전", "최악",
    "최고", "반드시", "무조건", "끝", "다시는", "영원히",
    "하나도", "매번", "늘", "절대로",
]


def count_words_from_list(text, word_list):
    return sum(str(text).count(word) for word in word_list)


analysis_df["char_len"] = analysis_df["clean_text"].str.len()
analysis_df["word_len"] = analysis_df["clean_text"].apply(lambda x: len(x.split()))

analysis_df["negative_count"] = analysis_df["clean_text"].apply(
    lambda x: count_words_from_list(x, negative_words)
)

analysis_df["emotion_count"] = analysis_df["clean_text"].apply(
    lambda x: count_words_from_list(x, emotion_words)
)

analysis_df["extreme_count"] = analysis_df["clean_text"].apply(
    lambda x: count_words_from_list(x, extreme_words)
)

feature_cols = ["char_len", "word_len", "negative_count", "emotion_count", "extreme_count"]

print("\n언어 특성 변수 생성 완료")
display(analysis_df[feature_cols].describe())




In [ ]:
# ============================================================
# 5. 생성 방식별 인지왜곡 유형 분포 분석


In [ ]:
# ============================================================

method_distortion_count = pd.crosstab(analysis_df[METHOD_COL], analysis_df[LABEL_COL])

method_distortion_ratio = pd.crosstab(
    analysis_df[METHOD_COL],
    analysis_df[LABEL_COL],
    normalize="index",
) * 100

chi2, p, dof, expected = chi2_contingency(method_distortion_count)

n = method_distortion_count.to_numpy().sum()
r, k = method_distortion_count.shape
cramers_v = np.sqrt(chi2 / (n * (min(r - 1, k - 1))))

print("\n[생성 방식 × 인지왜곡 유형] 카이제곱 검정")
print("chi-square:", chi2)
print("p-value:", p)
print("dof:", dof)
print("Cramer's V:", cramers_v)

method_distortion_count.to_csv(
    RESULT_DIR / "table_generation_method_distortion_count.csv",
    encoding="utf-8-sig",
)

method_distortion_ratio.to_csv(
    RESULT_DIR / "table_generation_method_distortion_ratio.csv",
    encoding="utf-8-sig",
)




In [ ]:
# ============================================================
# 6. 생성 모델별 언어 특성 평균


In [ ]:
# ============================================================

model_feature_mean = analysis_df.groupby(MODEL_COL)[feature_cols].mean()
model_feature_std = analysis_df.groupby(MODEL_COL)[feature_cols].std()

print("\n[생성 모델별 언어 특성 평균]")
display(model_feature_mean)

model_feature_mean.to_csv(RESULT_DIR / "table_model_feature_mean.csv", encoding="utf-8-sig")
model_feature_std.to_csv(RESULT_DIR / "table_model_feature_std.csv", encoding="utf-8-sig")




In [ ]:
# ============================================================
# 7. Kruskal-Wallis 검정


In [ ]:
# ============================================================

def epsilon_squared(H, n, k):
    return (H - k + 1) / (n - k)


kw_results = []
models = ["Claude", "GPT", "Gemini"]

for feature in feature_cols:
    groups = [
        analysis_df.loc[analysis_df[MODEL_COL] == model, feature].dropna()
        for model in models
    ]

    H, p_value = kruskal(*groups)
    eps2 = epsilon_squared(H, len(analysis_df), len(models))

    kw_results.append(
        {
            "variable": feature,
            "H": H,
            "p_value": p_value,
            "epsilon_squared": eps2,
        }
    )

kw_results_df = pd.DataFrame(kw_results)

print("\n[Kruskal-Wallis 검정 결과]")
display(kw_results_df)

kw_results_df.to_csv(
    RESULT_DIR / "table_kruskal_wallis_results.csv",
    index=False,
    encoding="utf-8-sig",
)




In [ ]:
# ============================================================
# 8. Random Forest 생성 모델 예측


In [ ]:
# ============================================================

X = analysis_df[feature_cols]
y = analysis_df[MODEL_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\n[Random Forest 생성 모델 예측]")
print("Accuracy:", accuracy)
print("\nClassification Report")
print(classification_report(y_test, y_pred))

conf_mat = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=models),
    index=[f"true_{m}" for m in models],
    columns=[f"pred_{m}" for m in models],
)

display(conf_mat)

conf_mat.to_csv(RESULT_DIR / "table_random_forest_confusion_matrix.csv", encoding="utf-8-sig")

feature_importance = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance": rf_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

print("\n[변수 중요도]")
display(feature_importance)

feature_importance.to_csv(
    RESULT_DIR / "table_random_forest_feature_importance.csv",
    index=False,
    encoding="utf-8-sig",
)




In [ ]:
# ============================================================
# 9. 그림 저장


In [ ]:
# ============================================================

plt.rcParams["axes.unicode_minus"] = False

plot_df = method_distortion_ratio.T
ax = plot_df.plot(kind="bar", figsize=(14, 6))
plt.title("Generation Method and Cognitive Distortion Distribution")
plt.xlabel("Cognitive Distortion Type")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Generation Method")
plt.tight_layout()
plt.savefig(RESULT_DIR / "figure1_generation_method_distortion.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 6))
model_feature_mean["char_len"].plot(kind="bar")
plt.title("Mean Character Length by LLM Model")
plt.xlabel("LLM Model")
plt.ylabel("Mean Character Length")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULT_DIR / "figure2_model_char_length.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
fi = feature_importance.sort_values("importance")
plt.barh(fi["feature"], fi["importance"])
plt.title("Feature Importance for Predicting LLM Models")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(RESULT_DIR / "figure3_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()




In [ ]:
# ============================================================
# 10. 핵심 결과 요약 및 ZIP 생성


In [ ]:
# ============================================================

summary = {
    "number_of_instances": len(analysis_df),
    "chi_square": chi2,
    "chi_square_p_value": p,
    "cramers_v": cramers_v,
    "random_forest_accuracy": accuracy,
    "top_features": feature_importance["feature"].head(2).tolist(),
}

summary_df = pd.DataFrame([summary])
display(summary_df)

summary_df.to_csv(
    RESULT_DIR / "summary_main_results.csv",
    index=False,
    encoding="utf-8-sig",
)

zip_output = Path("/content/KoACD_analysis_results.zip")

with zipfile.ZipFile(zip_output, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(RESULT_DIR):
        for file in files:
            file_path = Path(root) / file
            arcname = file_path.relative_to(RESULT_DIR)
            zipf.write(file_path, arcname)

print(f"\n결과 파일 압축 완료: {zip_output}")
print("분석 완료")

try:
    from google.colab import files
    files.download(str(zip_output))
except Exception:
    print("Colab이 아니면 직접 파일을 다운로드하세요:", zip_output)
